# Итоговый проект: «От дерева решений к ансамблю и кластеризации»

## Задание

**Проект** поможет увидеть, как на практике решаются задачи анализа данных — от простых правил до более сложных алгоритмов.

Вы научитесь строить алгоритмы, которые сами находят закономерности в данных, и поймёте, как такие методы применяются, например, при разделении клиентов на группы, прогнозе событий или оценке качества решений.

Проект даст вам опыт «взрослой» аналитической работы, когда нужно не просто обработать таблицу, а объяснить, почему модель принимает именно такие решения.

**Цель проекта**

Цель проекта — закрепить понимание принципов работы алгоритмов деревьев решений, ансамблей и кластеризации через собственную реализацию и анализ их поведения на данных.

В ходе проекта вы:

* реализуете базовые версии алгоритмов с нуля, используя numpy и pandas;
* проведёте эксперименты с параметрами и сравните результаты с реализациями из sklearn;
* научитесь интерпретировать результаты и визуализировать их.

**Формат выполнения**
1. Проект выполняется индивидуально.
2. Результат — Jupyter Notebook (.ipynb):
    * с описанием цели и задач проекта;
    * кодом с комментариями;
    * визуализациями и таблицами;
    * краткими выводами по каждому этапу.
3. В конце ноутбука добавьте общий вывод: чему вы научились и какие особенности алгоритмов заметили.

## Этапы проекта

### Этап 1. Реализация дерева решений (15 баллов)
На этом этапе вы создаёте простую реализацию дерева решений для задачи классификации.

#### Что нужно сделать

1. Сгенерируйте небольшой набор данных (100–300 наблюдений) с помощью функции:

```from sklearn.datasets import make_classification
X, y = make_classification(
    n_samples=200,
    n_features=2,  # два признака для наглядной визуализации границ решений
    n_informative=2,
    n_redundant=0,
    random_state=42
)
```
2. Преобразуйте данные в pandas.DataFrame.

3. Реализуйте функции:

* расчёта критерия Джини или энтропии;
* поиска наилучшего разбиения по признаку;
* рекурсивного построения дерева с ограничением по глубине (max_depth) или по числу объектов в узле (min_samples_split);
* предсказания для новых наблюдений.


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

In [14]:
# 1. Генерация данных
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    random_state=42,
    n_classes=2 # Явно указываем 2 класса для бинарной классификации
)

# 2. Преобразование в pandas.DataFrame
# Объединяем признаки X и целевую переменную y в один DataFrame
df = pd.DataFrame(X, columns=['feature_1', 'feature_2'])
df['target'] = y
display(df)

,feature_1,feature_2,target
0,1.689767,-1.408241,1
1,1.530287,-1.459848,1
2,-1.175042,-1.447633,0
3,-2.585395,0.963532,0
4,1.372246,0.440695,1
...,...,...,...
195,-0.435396,0.715716,0
196,1.040417,1.108613,1
197,1.883798,0.782433,0
198,1.829367,1.542978,1


In [11]:
display(df.target.value_counts())

target
1    100
0    100
Name: count, dtype: int64

In [ ]:
print(f"Размер сгенерированного набора данных: {df.shape}")
print(df.head())

# --- Реализация дерева решений ---

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    # 3.1. Расчет критерия Джини
    @staticmethod
    def _gini_impurity(y):
        """Рассчитывает критерий Джини для заданного набора меток."""
        if len(y) == 0:
            return 0
        p = np.bincount(y) / len(y)
        # Сумма квадратов вероятностей для каждого класса
        gini = 1.0 - np.sum(p**2)
        return gini

    # 3.2. Поиск наилучшего разбиения по признаку
    def _find_best_split(self, X, y):
        """Находит наилучшее пороговое значение и признак для разбиения данных."""
        best_gini = 1.0
        best_split = None
        n_samples, n_features = X.shape

        if n_samples < self.min_samples_split:
            return None

        current_gini = self._gini_impurity(y)

        for feature_idx in range(n_features):
            feature_values = X[:, feature_idx]
            # Уникальные значения признака в качестве возможных порогов
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                # Разделение данных на левый и правый поднаборы
                left_mask = feature_values <= threshold
                right_mask = feature_values > threshold
                
                # Избегаем пустых разбиений
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                # Расчет взвешенного Gini после разбиения
                gini_left = self._gini_impurity(y_left)
                gini_right = self._gini_impurity(y_right)
                
                weight_left = len(y_left) / n_samples
                weight_right = len(y_right) / n_samples
                gini_after_split = weight_left * gini_left + weight_right * gini_right

                # Расчет снижения неопределенности (Information Gain)
                information_gain = current_gini - gini_after_split

                if information_gain > 0 and gini_after_split < best_gini:
                    best_gini = gini_after_split
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }

        return best_split

    # 3.3. Рекурсивное построение дерева
    def _build_tree_recursive(self, X, y, depth=0):
        """Рекурсивно строит дерево решений."""
        
        # Условие останова 1: достигнута максимальная глубина
        if self.max_depth is not None and depth >= self.max_depth:
            # Превращаем узел в лист, предсказывая самый частый класс
            return {'leaf': True, 'class': np.bincount(y).argmax()}

        # Условие останова 2: недостаточно объектов для разбиения
        if len(y) < self.min_samples_split:
             return {'leaf': True, 'class': np.bincount(y).argmax()}

        split = self._find_best_split(X, y)
        
        # Условие останова 3: невозможно найти хорошее разбиение (все объекты одного класса или признаки неразделимы)
        if split is None:
            return {'leaf': True, 'class': np.bincount(y).argmax()}

        # Если разбиение найдено, создаем внутренний узел и продолжаем рекурсию
        left_branch = self._build_tree_recursive(X[split['left_mask']], y[split['left_mask']], depth + 1)
        right_branch = self._build_tree_recursive(X[split['right_mask']], y[split['right_mask']], depth + 1)

        return {
            'leaf': False,
            'feature_idx': split['feature_idx'],
            'threshold': split['threshold'],
            'left': left_branch,
            'right': right_branch
        }

    def fit(self, X, y):
        """Запускает процесс обучения дерева."""
        # Если X это DataFrame, преобразуем его в numpy array
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
            
        self.tree = self._build_tree_recursive(X, y, depth=0)
        return self

    # 3.4. Предсказание для новых наблюдений
    def _predict_one_sample(self, sample, node):
        """Проходит по дереву для одного образца."""
        if node['leaf']:
            return node['class']
        
        # Проверяем значение признака образца относительно порога узла
        if sample[node['feature_idx']] <= node['threshold']:
            return self._predict_one_sample(sample, node['left'])
        else:
            return self._predict_one_sample(sample, node['right'])

    def predict(self, X):
        """Делает предсказания для набора наблюдений."""
        if self.tree is None:
            raise Exception("Модель не обучена. Вызовите метод fit первым.")
            
        # Если X это DataFrame, преобразуем его в numpy array
        if isinstance(X, pd.DataFrame):
            X = X.values

        predictions = [self._predict_one_sample(sample, self.tree) for sample in X]
        return np.array(predictions)

# --- Использование реализованного дерева ---

# Разделяем данные на обучающую и тестовую выборки вручную
# (В реальных задачах лучше использовать train_test_split из sklearn)
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

X_train = train_df[['feature_1', 'feature_2']]
y_train = train_df['target']
X_test = test_df[['feature_1', 'feature_2']]
y_test = test_df['target']

# Инициализация и обучение дерева с ограничением глубины
print("\nНачинаем обучение дерева...")
tree_classifier = DecisionTree(max_depth=5, min_samples_split=10)
tree_classifier.fit(X_train, y_train)
print("Обучение завершено.")

# Предсказание на тестовом наборе
predictions = tree_classifier.predict(X_test)

# Оценка точности (accuracy)
accuracy = np.mean(predictions == y_test.values)
print(f"\nПредсказания для первых 5 тестовых образцов: {predictions[:5]}")
print(f"Фактические метки для первых 5 тестовых образцов: {y_test.values[:5]}")
print(f"Точность (Accuracy) на тестовой выборке: {accuracy:.4f}")

Размер сгенерированного набора данных: (200, 3)
   feature_1  feature_2  target
0   1.689767  -1.408241       1
1   1.530287  -1.459848       1
2  -1.175042  -1.447633       0
3  -2.585395   0.963532       0
4   1.372246   0.440695       1

Начинаем обучение дерева...
Обучение завершено.

Предсказания для первых 5 тестовых образцов: [1 1 0 0 1]
Фактические метки для первых 5 тестовых образцов: [1 1 0 0 1]
Точность (Accuracy) на тестовой выборке: 0.8750
